# Day 3: 数据处理与批次管理

## 本节目标

1. 理解 Istanbul 数据集格式
2. 掌握 Sequence 类的实现
3. 理解 Batch 类的 padding 逻辑
4. 学习多数据集适配方法

---

## Part 1: Istanbul 数据集格式

### 1.1 数据集概览

**Istanbul 数据集**是 Marionette 使用的轨迹数据集，记录了用户在伊斯坦布尔的移动轨迹。

#### 数据规模
- **POI 数量**: 381 个独特的兴趣点
- **类别数量**: 9 个 POI 类别
- **时间段**: 24 小时分段，每小时一个时段

### 1.2 数据结构

数据存储在 `.pkl` 文件中，包含以下结构：

In [ ]:
import pickle
import numpy as np

# 模拟 Istanbul 数据集的结构
def show_dataset_structure():
    """显示数据集结构"""
    
    # 单个序列的示例
    example_sequence = {
        'arrival_times': np.array([3600, 7200, 14400, 28800]),  # 到达时间戳（秒）
        'marks': np.array([0, 2, 1, 0]),  # POI 类别 ID
        'checkins': np.array([10, 45, 23, 10]),  # POI ID
        'gps': [[41.0082, 28.9784], [41.0123, 28.9821],  # GPS 坐标
                [41.0056, 28.9756], [41.0082, 28.9784]],
        
        # 条件变量（6 种条件类型）
        'condition1': np.array([25]),  # 星期几 (25-31: 周一到周日)
        'condition2': np.array([32]),  # 斋月 (32: 非斋月, 33: 斋月)
        'condition3': np.array([34]),  # 公共假期 (34: 非假期, 35: 假期)
        'condition4': np.array([0]),   # 未使用 (常数值)
        'condition5': np.array([0]),   # 未使用 (常数值)
        'condition6': np.array([0]),   # 未使用 (常数值)
        
        # 条件指示器（精细粒度）
        'condition1_indicator': np.eye(24)[9:10],  # 24 维时间段指示器
        'condition2_indicator': np.array([[1, 0]]),  # 斋月指示器
        'condition3_indicator': np.array([[1, 0]]),  # 假期指示器
        'condition4_indicator': np.array([[1]]),     # 未使用指示器
        'condition5_indicator': np.array([[1]]),     # 未使用指示器
        'condition6_indicator': np.array([[1]]),     # 未使用指示器
    }
    
    # 完整的数据集结构
    dataset = {
        'sequences': [example_sequence],  # 用户序列列表
        't_max': 86400.0,  # 序列最大时间（24小时 = 86400秒）
        'num_marks': 9,  # POI 类别数量
        'num_pois': 381,  # POI 总数
        'poi_gps': {  # POI 到 GPS 坐标的映射
            10: [41.0082, 28.9784],
            23: [41.0056, 28.9756],
            45: [41.0123, 28.9821],
        }
    }
    
    return dataset, example_sequence

dataset, example_sequence = show_dataset_structure()

print("=== Istanbul 数据集结构 ===")
print(f"\n数据集元数据:")
print(f"  t_max (最大时间): {dataset['t_max']} 秒")
print(f"  num_marks (类别数): {dataset['num_marks']}")
print(f"  num_pois (POI总数): {dataset['num_pois']}")

print(f"\n单个序列字段:")
for key, value in example_sequence.items():
    if isinstance(value, np.ndarray):
        print(f"  {key}: shape={value.shape}, dtype={value.dtype}")
    else:
        print(f"  {key}: {type(value).__name__}")

### 1.3 字段含义详解

In [ ]:
# 详细说明每个字段的含义

print("=== 字段含义详解 ===")

field_descriptions = {
    "arrival_times": "事件发生的绝对时间戳（从序列开始计算的秒数）",
    "marks": "POI 类别 ID（0-8，共9个类别）",
    "checkins": "具体的 POI ID（0-380，共381个位置）",
    "gps": "POI 的 GPS 坐标 [纬度, 经度]，用于可视化",
    
    "condition1": "星期几: 25=周一, 26=周二, ..., 31=周日",
    "condition2": "斋月状态: 32=非斋月, 33=斋月",
    "condition3": "假期状态: 34=非假期, 35=假期",
    "condition4-6": "预留条件字段（未使用）",
    
    "condition1_indicator": "24 维时间段指示器（每一维代表一个小时）",
    "condition2_indicator": "斋月指示器（2 维）",
    "condition3_indicator": "假期指示器（2 维）",
    "condition4-6_indicator": "预留指示器（未使用）",
}

for field, desc in field_descriptions.items():
    print(f"\n【{field}】")
    print(f"  {desc}")

---

## Part 2: Sequence 类深度解析

Sequence 类是 Marionette 中表示单个用户轨迹的核心数据结构。

### 2.1 Sequence 类的结构

In [ ]:
import torch

class Sequence:
    """
    表示单个用户轨迹序列
    
    核心功能:
    1. 存储轨迹的所有信息（时间、空间、条件）
    2. 计算时间间隔 (tau)
    3. 支持设备切换（CPU/GPU）
    """
    
    def __init__(
        self,
        time,          # 到达时间 [seq_len]
        condition1,    # 条件1 [seq_len]
        condition2,    # 条件2 [seq_len]
        condition3,    # 条件3 [seq_len]
        condition4,    # 条件4 [seq_len]
        condition5,    # 条件5 [seq_len]
        condition6,    # 条件6 [seq_len]
        condition1_indicator,  # 条件1指示器 [granularity]
        condition2_indicator,  # 条件2指示器 [granularity]
        condition3_indicator,  # 条件3指示器 [granularity]
        condition4_indicator,  # 条件4指示器 [granularity]
        condition5_indicator,  # 条件5指示器 [granularity]
        condition6_indicator,  # 条件6指示器 [granularity]
        tmax,          # 最大时间（序列长度）
        checkins=None, # POI ID [seq_len]
        category=None  # POI 类别 [seq_len]
    ):
        # 存储所有字段
        self.time = time
        self.condition1 = condition1
        self.condition2 = condition2
        self.condition3 = condition3
        self.condition4 = condition4
        self.condition5 = condition5
        self.condition6 = condition6
        self.condition1_indicator = condition1_indicator
        self.condition2_indicator = condition2_indicator
        self.condition3_indicator = condition3_indicator
        self.condition4_indicator = condition4_indicator
        self.condition5_indicator = condition5_indicator
        self.condition6_indicator = condition6_indicator
        self.tmax = tmax
        self.checkins = checkins
        self.category = category
        
        # 计算时间间隔 tau
        self._compute_tau()
    
    def _compute_tau(self):
        """
        计算时间间隔 tau
        
        tau[i] = time[i] - time[i-1]
        tau[0] = 0 - time[0] = -time[0]
        tau[-1] = tmax - time[-1]
        
        输入: time = [3600, 7200, 14400, 28800]
        输出: tau = [-3600, 3600, 7200, 14400, 57600]
              ^      ^     ^     ^      ^
              |      |     |     |      |
             0-start 1-0   2-1   3-2   end-3
        """
        self.tau = torch.diff(
            self.time,
            prepend=torch.as_tensor([0.0], device=self.time.device),
            append=torch.as_tensor([self.tmax], device=self.time.device),
        )
    
    def to(self, device):
        """将所有数据移动到指定设备（CPU/GPU）"""
        self.time = self.time.to(device)
        self.tau = self.tau.to(device)
        # ... 其他字段同样处理
        return self

# 创建示例 Sequence
seq = Sequence(
    time=torch.tensor([3600.0, 7200.0, 14400.0, 28800.0]),
    condition1=torch.tensor([25, 25, 25, 25]),
    condition2=torch.tensor([32, 32, 32, 32]),
    condition3=torch.tensor([34, 34, 34, 34]),
    condition4=torch.tensor([0, 0, 0, 0]),
    condition5=torch.tensor([0, 0, 0, 0]),
    condition6=torch.tensor([0, 0, 0, 0]),
    condition1_indicator=torch.eye(24)[9:10],
    condition2_indicator=torch.tensor([[1, 0]]),
    condition3_indicator=torch.tensor([[1, 0]]),
    condition4_indicator=torch.tensor([[1]]),
    condition5_indicator=torch.tensor([[1]]),
    condition6_indicator=torch.tensor([[1]]),
    tmax=86400.0,
    checkins=torch.tensor([10, 45, 23, 10]),
    category=torch.tensor([0, 2, 1, 0])
)

print("=== Sequence 示例 ===")
print(f"\ntime: {seq.time}")
print(f"tau (时间间隔): {seq.tau}")
print(f"checkins: {seq.checkins}")
print(f"category: {seq.category}")

### 2.2 tau 计算详解

In [ ]:
# 详细演示 tau 的计算过程

print("=== tau 计算详解 ===")

# 原始时间序列
time = torch.tensor([3600.0, 7200.0, 14400.0, 28800.0])
tmax = 86400.0

print(f"原始时间: {time}")
print(f"tmax: {tmax}")

# 计算 diff
print(f"\n步骤 1: prepend 0")
prepended = torch.cat([torch.tensor([0.0]), time])
print(f"  {prepended}")

print(f"\n步骤 2: append tmax")
appended = torch.cat([prepended, torch.tensor([tmax])])
print(f"  {appended}")

print(f"\n步骤 3: 计算差分")
tau = torch.diff(appended)
print(f"  tau = {tau}")

print(f"\n解释:")
print(f"  tau[0] = {tau[0]} (第1个事件前的时间间隔)")
print(f"  tau[1] = {tau[1]} (事件1和事件2之间的间隔)")
print(f"  tau[2] = {tau[2]} (事件2和事件3之间的间隔)")
print(f"  tau[3] = {tau[3]} (事件3和事件4之间的间隔)")
print(f"  tau[4] = {tau[4]} (最后1个事件后的时间间隔)")

---

## Part 3: Batch 类深度解析

Batch 类负责将多个 Sequence 组织成一个批次，用于批量训练。

### 3.1 Batch 的核心挑战

**问题**：不同用户的轨迹长度不同，无法直接组合成批次。

**解决方案**：Padding + Masking
- Padding：填充短序列，使所有序列长度相同
- Masking：标记哪些是真实数据，哪些是填充值

In [ ]:
def pad_sequence(sequences, batch_first=True, padding_value=0):
    """
    填充变长序列
    
    输入: List[Tensor], 每个张量形状不同
    输出: Tensor, 所有序列填充到相同长度
    
    示例:
        输入: [tensor([1, 2, 3]), tensor([4, 5]), tensor([6, 7, 8, 9])]
        输出: tensor([[1, 2, 3, 0],
                     [4, 5, 0, 0],
                     [6, 7, 8, 9]])
    """
    # 找到最大长度
    max_len = max(seq.size(-1) for seq in sequences)
    
    # 创建输出张量
    if batch_first:
        out_dims = (len(sequences), max_len)
    else:
        out_dims = (max_len, len(sequences))
    
    output = torch.full(out_dims, padding_value, dtype=sequences[0].dtype)
    
    # 填充
    for i, seq in enumerate(sequences):
        length = seq.size(-1)
        if batch_first:
            output[i, :length] = seq
        else:
            output[:length, i] = seq
    
    return output

# 示例
sequences = [
    torch.tensor([1, 2, 3]),
    torch.tensor([4, 5]),
    torch.tensor([6, 7, 8, 9])
]

padded = pad_sequence(sequences, batch_first=True, padding_value=0)

print("=== Padding 示例 ===")
print("原始序列:")
for i, seq in enumerate(sequences):
    print(f"  序列 {i}: {seq}")

print(f"\nPadding 后:\n{padded}")

### 3.2 Batch 类的实现

In [ ]:
class Batch:
    """
    批次数据封装
    
    核心功能:
    1. 将多个 Sequence 组合成批次
    2. 自动 padding 和 masking
    3. 支持设备切换
    4. 支持反向转换为序列列表
    """
    
    @staticmethod
    def from_sequence_list(sequences):
        """
        从 Sequence 列表创建 Batch
        
        输入: List[Sequence]
        输出: Batch
        """
        # 1. 填充 tau 到相同长度
        tau = pad_sequence([s.tau for s in sequences], batch_first=True, padding_value=0)
        
        # 2. 填充 time
        time = pad_sequence([s.time for s in sequences], batch_first=True, padding_value=0)
        
        # 3. 填充所有条件
        condition1 = pad_sequence([s.condition1 for s in sequences], batch_first=True, padding_value=0)
        # ... 其他条件同样处理
        
        # 4. 计算序列长度
        sequence_length = torch.tensor([len(s.time) for s in sequences])
        
        # 5. 创建掩码
        mask = (
            torch.arange(0, tau.shape[-1], device=tau.device)[None, :] 
            < sequence_length[:, None]
        )
        
        # 6. 构建 Batch 对象
        batch = Batch()
        batch.tau = tau
        batch.time = time
        batch.condition1 = condition1
        # ... 其他字段
        batch.mask = mask
        batch.batch_size = len(sequences)
        
        return batch

# 创建示例 Batch
seq1 = Sequence(
    time=torch.tensor([3600.0, 7200.0, 14400.0]),
    condition1=torch.tensor([25, 25, 25]),
    condition2=torch.tensor([32, 32, 32]),
    condition3=torch.tensor([34, 34, 34]),
    condition4=torch.tensor([0, 0, 0]),
    condition5=torch.tensor([0, 0, 0]),
    condition6=torch.tensor([0, 0, 0]),
    condition1_indicator=torch.eye(24)[9:10],
    condition2_indicator=torch.tensor([[1, 0]]),
    condition3_indicator=torch.tensor([[1, 0]]),
    condition4_indicator=torch.tensor([[1]]),
    condition5_indicator=torch.tensor([[1]]),
    condition6_indicator=torch.tensor([[1]]),
    tmax=86400.0,
    checkins=torch.tensor([10, 45, 23]),
    category=torch.tensor([0, 2, 1])
)

seq2 = Sequence(
    time=torch.tensor([5400.0, 10800.0]),
    condition1=torch.tensor([26, 26]),
    condition2=torch.tensor([32, 32]),
    condition3=torch.tensor([34, 34]),
    condition4=torch.tensor([0, 0]),
    condition5=torch.tensor([0, 0]),
    condition6=torch.tensor([0, 0]),
    condition1_indicator=torch.eye(24)[10:11],
    condition2_indicator=torch.tensor([[1, 0]]),
    condition3_indicator=torch.tensor([[1, 0]]),
    condition4_indicator=torch.tensor([[1]]),
    condition5_indicator=torch.tensor([[1]]),
    condition6_indicator=torch.tensor([[1]]),
    tmax=86400.0,
    checkins=torch.tensor([15, 30]),
    category=torch.tensor([1, 0])
)

batch = Batch.from_sequence_list([seq1, seq2])

print("=== Batch 示例 ===")
print(f"\ntau (已padding):\n{batch.tau}")
print(f"\ntime (已padding):\n{batch.time}")
print(f"\nmask (1=有效, 0=填充):\n{batch.mask}")

---

## Part 4: 多数据集适配

### 4.1 适配 Checklist

要将 Marionette 适配到新数据集，需要检查以下方面：

In [ ]:
# 新数据集适配 Checklist

print("=== 新数据集适配 Checklist ===")

checklist = [
    ("1. 数据格式检查", [
        "确保数据集包含必要的字段",
        "  - sequences: 用户序列列表",
        "  - t_max: 序列最大时间",
        "  - num_marks: POI 类别数量",
        "  - num_pois: POI 总数",
        "  - poi_gps: POI 到 GPS 的映射"
    ]),
    ("2. 条件字段适配", [
        "确定数据集的条件类型数量",
        "调整 Sequence 类中的条件字段数量",
        "更新 Batch 类中的条件处理逻辑",
        "修改 conditionX_indicator 的维度"
    ]),
    ("3. 数据规模调整", [
        "更新 num_category 和 num_poi 以匹配新数据集",
        "调整 temporal_time_segments 参数",
        "修改 alpha_schedule 中的类别数量参数"
    ]),
    ("4. 模型配置调整", [
        "更新 config/model/Marionette.yaml:",
        "  - num_condition_types",
        "  - type_classes",
        "  - poi_classes",
        "  - temporal_time_segments"
    ]),
    ("5. 评估指标适配", [
        "确保 GPS 坐标映射存在",
        "调整评估指标以适应新数据集的特征"
    ])
]

for section, items in checklist:
    print(f"\n【{section}】")
    for item in items:
        print(f"  ✓ {item}")

### 4.2 条件字段扩展示例

假设新数据集有 7 种条件类型（而不是 6 种），需要修改：

In [ ]:
# 示例：添加 condition7

print("=== 添加 condition7 的修改点 ===")

modifications = [
    ("datamodule.py - Sequence 类", [
        "def __init__(self, ..., condition7, condition7_indicator, ...):",
        "    self.condition7 = condition7",
        "    self.condition7_indicator = condition7_indicator"
    ]),
    ("datamodule.py - Batch.from_sequence_list", [
        "batch.condition7 = pad_sequence([s.condition7 for s in sequences])",
        "batch.condition7_indicator = torch.stack([s.condition7_indicator for s in sequences])"
    ]),
    ("config/model/Marionette.yaml", [
        "num_condition_types: 7  # 从 6 改为 7"
    ]),
    ("add_thin/diffusion/model.py", [
        "确保条件编码器能处理 7 种条件类型"
    ]),
    ("discrete_diffusion/diffusion_transformer.py", [
        "确保 Transformer 能处理 7 种条件嵌入"
    ])
]

for location, changes in modifications:
    print(f"\n【{location}】")
    for change in changes:
        print(f"  - {change}")

---

## Part 5: 实践练习

### 练习 1: 编写函数打印 Sequence 的所有字段

In [ ]:
# 练习 1: Sequence 字段打印函数

def print_sequence_info(seq):
    """
    TODO: 打印 Sequence 的所有字段信息
    
    要求:
    1. 打印每个字段的名称、形状、类型
    2. 打印每个字段的前 5 个值
    3. 格式化输出，易于阅读
    """
    # 实现你的代码
    pass

# 测试
print("请实现 print_sequence_info 函数...")

### 练习 2: 实现 Batch 的 to_seq_list 方法

In [ ]:
# 练习 2: 反向转换 Batch -> Sequence 列表

def to_seq_list(self, gps_dict=None):
    """
    TODO: 将 Batch 转换回 Sequence 列表
    
    要求:
    1. 根据 mask 移除填充值
    2. 恢复每个序列的原始长度
    3. 返回 List[dict]，每个 dict 包含序列的所有字段
    """
    # 实现你的代码
    pass

# Batch.to_seq_list = to_seq_list  # 添加方法到 Batch 类

print("请实现 to_seq_list 方法...")

### 练习 3: 设计新数据集的格式转换脚本

In [ ]:
# 练习 3: 数据集格式转换

def convert_to_marionette_format(raw_data_path, output_path):
    """
    TODO: 将原始数据转换为 Marionette 格式
    
    输入: raw_data_path - 原始数据文件路径
    输出: output_path - 输出的 .pkl 文件路径
    
    要求:
    1. 读取原始数据（假设是 CSV 格式）
    2. 提取时间、POI、条件等字段
    3. 构建符合 Marionette 格式的数据集
    4. 保存为 .pkl 文件
    """
    # 实现你的代码
    pass

print("请实现数据集格式转换脚本...")

---

## Day 3 总结

### 今天学到了什么？

1. **Istanbul 数据集格式**: 理解了轨迹数据的存储结构
2. **Sequence 类**: 掌握了单条轨迹的表示方法和 tau 计算
3. **Batch 类**: 理解了 padding 和 masking 的原理
4. **多数据集适配**: 学会了如何适配新数据集的流程

### 关键概念总结

| 概念 | 作用 | 输入 | 输出 |
|------|------|------|------|
| **tau** | 时间间隔 | time 数组 | tau 数组 |
| **padding** | 填充变长序列 | 不同长度的序列 | 相同长度的批次 |
| **mask** | 标记有效数据 | 序列长度 | 布尔掩码 |
| **condition** | 控制生成条件 | 条件值 | 条件嵌入 |

### 明天预告

Day 4 将学习 **扩散模型核心原理**，包括：
- 扩散模型的直观理解
- Add-THIN 时间扩散模型
- 离散扩散空间模型
- 条件生成机制

### 课后作业

1. 完成 3 个练习题
2. 思考：为什么 tau 的计算要 prepend 0 和 append tmax？
3. 预习扩散模型的基本概念

---

**📚 Day 3 概念测验**

1. tau（时间间隔）是如何计算的？

2. 为什么要对序列进行 padding？

3. mask 的作用是什么？

请将答案记录在笔记本中，明天我们将在课程开始时讨论。